# YOLO2D+t Colab Training Notebook

Notebook nay chay baseline train cho YOLOv1-style 2D+t voi dataset dang nam tren Google Drive.

Flow:
1. Cai dependency
2. Mount Google Drive
3. Copy `mot17_2dt_dataset.py` vao repo
4. Tao config train phu hop voi path tren Drive
5. Chay train


In [ ]:
from pathlib import Path

REPO_DIR = Path('/content/YOLO2D-t')
if not REPO_DIR.exists():
    !git clone https://github.com/thangSy221105/YOLO2D-t.git /content/YOLO2D-t
%cd /content/YOLO2D-t


In [ ]:
!pip install -q -r requirements.txt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

# Sua 3 path nay neu cau truc Drive cua ban khac.
DRIVE_ROOT = Path('/content/drive/MyDrive/dataset_YOLO/YOLO 2D+t')
DATASET_SCRIPT_PATH = DRIVE_ROOT / 'scripts' / 'mot17_2dt_dataset.py'
PROCESSED_DIR = DRIVE_ROOT / 'data' / 'processed'

print('DATASET_SCRIPT_PATH =', DATASET_SCRIPT_PATH)
print('PROCESSED_DIR =', PROCESSED_DIR)
print('dataset script exists =', DATASET_SCRIPT_PATH.exists())
print('processed dir exists =', PROCESSED_DIR.exists())


In [ ]:
!mkdir -p scripts
!cp "{DATASET_SCRIPT_PATH}" scripts/mot17_2dt_dataset.py
!ls scripts


In [ ]:
from pathlib import Path
import textwrap

config_text = f"""
seed: 42

data:
  image_size: 448
  grid_size: 7
  boxes_per_cell: 2
  num_classes: 1
  batch_size: 4
  num_workers: 2
  delta: 40
  processed_dir: {PROCESSED_DIR.as_posix()}
  train_split: train
  val_split: val
  use_horizontal_flip: false
  return_meta: false
  pin_memory: true

model:
  in_channels: 6
  hidden_dim: 512
  dropout: 0.1

loss:
  lambda_coord: 5.0
  lambda_noobj: 0.5
  lambda_class: 1.0
  lambda_motion: 1.0

train:
  epochs: 10
  lr: 1.0e-4
  weight_decay: 1.0e-4
  device: cuda
  mixed_precision: true
  grad_clip_norm: 10.0
  log_interval: 10
  output_dir: outputs/colab_run
  save_every_epoch: true
"""

config_path = Path('configs/colab_drive.yaml')
config_path.write_text(textwrap.dedent(config_text).strip() + '\n', encoding='utf-8')
print(config_path.read_text(encoding='utf-8'))


In [ ]:
import sys

sys.path.insert(0, 'src')

from yolo2dt.config import load_config
from yolo2dt.data_adapter import build_dataloaders

cfg = load_config('configs/colab_drive.yaml')
train_loader, val_loader = build_dataloaders(cfg)
batch = next(iter(train_loader))

if isinstance(batch, dict):
    print('image shape =', tuple(batch['image'].shape))
    print('target shape =', tuple(batch['target'].shape))
    print('motion_mask shape =', tuple(batch['motion_mask'].shape))
else:
    print('image shape =', tuple(batch[0].shape))
    print('target shape =', tuple(batch[1].shape))
    print('motion_mask shape =', tuple(batch[2].shape))


In [ ]:
!python train.py --config configs/colab_drive.yaml


In [ ]:
!find outputs/colab_run -maxdepth 2 -type f | sort
